# Продуктовая аналитика: Поиск точек роста бизнеса

**Ключевые показатели юнит-экономики:**
- **UA** (User Acquisition) — Общее количество входящих лидов.
- **C1** (Conversion Rate) — Конверсия из лида в уникального покупателя (B/UA).
- **B** (Buyers) — Количество уникальных подтверждённых покупателей.
- **T** (Transactions) — Общее количество подтверждённых сделок (оплат).
- **Rev** (Revenue) — Выручка (сумма начальных платежей подтверждённых покупателей).
- **APC** (Average Purchase Count) — Среднее количество покупок на одного покупателя (T/B).
- **AOV** (Average Order Value) — Средний чек за транзакцию (Rev/T).
- **AC** (Acquisition Cost) — Общие маркетинговые затраты.
- **CAC** (Customer Acquisition Cost) — Стоимость привлечения одного покупателя (AC/B).
- **CPA** (Cost Per Acquisition/Lead) — Стоимость одного лида (AC/UA). В коде часто `LTC`.
- **CLTV** (Customer Lifetime Value) — Доход от одного покупателя ($APC \times AOV$).
- **LTV** (Lifetime Value) — Доход от одного привлеченного лида ($C1 \times APC \times AOV = C1 \times CLTV$).
- **CM** (Contribution Margin) — Маржинальная прибыль (Rev - AC).

**Изменяемые рычаги для анализа чувствительности:** `UA`, `C1`, `APC`, `AOV`, `LTC`.


In [119]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from IPython.display import display

import help_130625_dam  as h

pd.set_option('display.float_format', '{:.2f}'.format)
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (14, 5)

CLEANED_DIR = os.path.join('..', 'data', 'cleaned')

In [120]:
contacts      = pd.read_pickle(os.path.join(CLEANED_DIR, 'contacts_clean.pkl'))
deals         = pd.read_pickle(os.path.join(CLEANED_DIR, 'deals_clean.pkl'))
calls         = pd.read_pickle(os.path.join(CLEANED_DIR, 'calls_clean.pkl'))
spend         = pd.read_pickle(os.path.join(CLEANED_DIR, 'spend_clean.pkl'))
campaign_romi = pd.read_pickle(os.path.join(CLEANED_DIR, 'campaign_romi.pkl'))

print(f'Contacts:      {contacts.shape}')
print(f'Deals:         {deals.shape}')
print(f'Calls:         {calls.shape}')
print(f'Spend:         {spend.shape}')
print(f'Campaign ROMI: {campaign_romi.shape}')

Contacts:      (18510, 7)
Deals:         (19815, 28)
Calls:         (92599, 9)
Spend:         (19862, 8)
Campaign ROMI: (366, 12)


## Глобальная юнит-экономика

Сначала смотрим на бизнес целиком: как соотносятся затраты и доходы на уровне одного лида/клиента.

In [121]:
#  Диагностика
paid_all    = deals[deals['initial_amount_paid'] > 0]
paid_buyers = deals[deals['is_buyer'] & (deals['initial_amount_paid'] > 0)]

print(' Диагностика расхождения Rev ')
print(f'Всего строк в deals:                          {len(deals):,}')
print(f'Строк с initial_amount_paid > 0:              {len(paid_all):,}')
print(f'  из них is_buyer=True:                       {len(paid_buyers):,}')
print(f'  из них is_buyer=False (аномалия):           {len(paid_all) - len(paid_buyers):,}')
print()
print(f'Revenue ALL  (paid > 0):                      {paid_all["initial_amount_paid"].sum():,.0f} €')
print(f'Revenue BUYER (paid > 0 & is_buyer):          {paid_buyers["initial_amount_paid"].sum():,.0f} €')
print(f'Revenue на "аномальных" строках:              {(paid_all["initial_amount_paid"].sum() - paid_buyers["initial_amount_paid"].sum()):,.0f} €')
print()

# Уникальные контакты в deals с оплатой, у которых is_buyer=False
anomaly_contacts = paid_all[~paid_all['is_buyer']]['contact_id'].nunique()
print(f'Уникальных contact_id с оплатой, но is_buyer=False: {anomaly_contacts:,}')
print()

# # Посмотрим на несколько аномальных строк
# anomaly_rows = paid_all[~paid_all['is_buyer']][['contact_id', 'initial_amount_paid', 'stage', 'is_buyer']].head(10)
# print('Пример аномальных строк (paid > 0, is_buyer=False):')
# display(anomaly_rows)

 Диагностика расхождения Rev 
Всего строк в deals:                          19,815
Строк с initial_amount_paid > 0:              3,225
  из них is_buyer=True:                       826
  из них is_buyer=False (аномалия):           2,399

Revenue ALL  (paid > 0):                      3,830,310 €
Revenue BUYER (paid > 0 & is_buyer):          941,850 €
Revenue на "аномальных" строках:              2,888,460 €

Уникальных contact_id с оплатой, но is_buyer=False: 2,374



In [ ]:
# Глобальные показатели юнит-экономики
n_leads  = len(deals)
# B = Уникальные контакты, ставшие покупателями
n_buyers = int(contacts['is_buyer'].sum())
crv_rate = n_buyers / n_leads if n_leads > 0 else 0

buyer_deals = deals[deals['is_buyer']]

# T = Общее число подтверждённых транзакций
n_transactions = len(buyer_deals)

total_revenue  = buyer_deals['initial_amount_paid'].sum()
total_spend    = spend['spend'].sum()

# Основные юнит-метрики
apc = n_transactions / n_buyers if n_buyers > 0 else 0
aov = total_revenue / n_transactions if n_transactions > 0 else 0
cac = total_spend / n_buyers if n_buyers > 0 else 0
ltc = total_spend / n_leads if n_leads > 0 else 0
cltv = apc * aov          
ltv = crv_rate * cltv      
profit = total_revenue - total_spend

global_ue = pd.DataFrame([{
    'UA':        n_leads,
    'C1, %':     crv_rate,
    'B':         n_buyers,
    'T':         n_transactions,
    'APC':       apc,
    'Rev, €':    total_revenue,
    'AOV, €':    aov,
    'AC, €':     total_spend,
    'CAC, €':    cac,
    'CPA, €':    ltc,
    'CLTV, €':   cltv,
    'LTV, €':    ltv,
    'CM, €':     profit,
}])

display(
    global_ue.style
    .hide(axis='index')
    .format({
        'UA':        '{:,.0f}',
        'C1, %':     '{:.2%}',
        'B':         '{:,.0f}',
        'T':         '{:,.0f}',
        'APC':       '{:.2f}',
        'Rev, €':    '{:,.0f}',
        'AOV, €':    '{:,.0f}',
        'AC, €':     '{:,.0f}',
        'CAC, €':    '{:,.0f}',
        'CPA, €':    '{:,.2f}',
        'CLTV, €':   '{:,.2f}',
        'LTV, €':    '{:,.2f}',
        'CM, €':     '{:,.0f}',
    })
    .map(
        lambda v: 'color: red; font-weight: bold' if isinstance(v, (int, float)) and v < 0 else '',
        subset=['CM, €']
    )
    .set_caption('Глобальная юнит-экономика')
)

print(f'   Retention: отсутствует. Бизнес-модель "одной продажи".')
print(f'   LTV/CAC (Lead-based) = {ltv / (ltc if ltc > 0 else 1):.2f}')
print(f'   CLTV/CAC (Customer-based) = {cltv / (cac if cac > 0 else 1):.2f}')


UA,"C1, %",B,T,APC,"Rev, €","AOV, €","AC, €","CAC, €","CPA, €","CLTV, €","LTV, €","CM, €"
"19,815",4.11%,814,826,1.01,"941,850","1,140","149,523",184,7.55,"1,157.06",47.53,"792,327"


   Retention: отсутствует. Бизнес-модель "одной продажи".
   LTV/CAC (Lead-based) = 6.30
   CLTV/CAC (Customer-based) = 6.30


#### Число транзакций на покупателя = 1.01 объясняется тем что после первой оплаты и начала обучения клиент становился неинтересен менеджеру и сделка закрывалась.

## Юнит-экономика по источникам трафика

In [122]:
# Юнит-экономика по продуктам (используем общие UA и AC)
# В анализе остаются только продукты с подтвержденными продажами (buyers > 0)

def product_agg(x):
    b = x[x['is_buyer']]['contact_id'].nunique()
    t = x['is_buyer'].sum()
    rev = x.loc[x['is_buyer'], 'initial_amount_paid'].sum()
    return pd.Series({
        'B':   b,
        'T':   t,
        'Rev': rev,
        'UA':  n_leads,
        'AC':  total_spend
    })

product_ue = (
    deals[deals['product'] != 'Unknown']
    .groupby('product', observed=True)
    .apply(product_agg, include_groups=False)
    .reset_index()
)

# Оставляем продукты с продажами
product_ue = product_ue[product_ue['B'] > 0].copy()

# Расчёт всех метрик
product_ue['c1']    = product_ue['B'] / product_ue['UA']
product_ue['apc']   = product_ue['T'] / product_ue['B']
product_ue['aov']   = product_ue['Rev'] / product_ue['T']
product_ue['cltv']  = product_ue['apc'] * product_ue['aov']        # Доход на покупателя (CLTV)
product_ue['ltv']   = product_ue['c1'] * product_ue['cltv']       # Доход на лид (LTV)
product_ue['cac']   = product_ue['AC'] / product_ue['B']
product_ue['cpa']   = product_ue['AC'] / product_ue['UA']
product_ue['cm']    = product_ue['Rev'] - product_ue['AC']

product_ue = product_ue.sort_values('Rev', ascending=False).set_index('product')

print(f'Справочно: Продуктов с продажами: {len(product_ue)} | Глобальный UA = {n_leads:,.0f} | Глобальный AC = {total_spend:,.0f} €')

display(
    product_ue[['UA', 'c1', 'B', 'aov', 'T', 'Rev', 'apc', 'AC', 'cac', 'cpa', 'cltv', 'ltv', 'cm']]
    .head(15)
    .rename(columns={
        'UA':    'UA',
        'c1':    'C1, %',
        'B':     'B',
        'aov':   'AOV, €',
        'T':     'T',
        'Rev':   'Rev, €',
        'apc':   'APC',
        'AC':    'AC, €',
        'cac':   'CAC, €',
        'cpa':   'CPA, €',
        'cltv':  'CLTV, €',
        'ltv':   'LTV, €',
        'cm':    'CM, €',
    })
    .style
    .format({
        'UA':      '{:,.0f}',
        'C1, %':   '{:.2%}',
        'B':       '{:,.0f}',
        'AOV, €':  '{:,.0f}',
        'T':       '{:,.0f}',
        'Rev, €':  '{:,.0f}',
        'APC':     '{:.2f}',
        'AC, €':   '{:,.0f}',
        'cac, €':  '{:,.0f}',
        'CPA, €':  '{:,.2f}',
        'CLTV, €': '{:,.2f}',
        'LTV, €':  '{:,.2f}',
        'CM, €':   '{:,.0f}',
    })
    .map(
        lambda v: 'color: red' if isinstance(v, (int, float)) and v < 0 else '',
        subset=['CM, €']
    )
    .set_caption('Юнит-экономика: Эффективность продуктов')
)


Справочно: Продуктов с продажами: 3 | Глобальный UA = 19,815 | Глобальный AC = 149,523 €


,UA,"C1, %",B,"AOV, €",T,"Rev, €",APC,"AC, €","CAC, €","CPA, €","CLTV, €","LTV, €","CM, €"
product,,,,,,,,,,,,,
Digital Marketing,"19,815",2.34%,463,"1,152",468,"539,100",1.01,"149,523",322.944816,7.55,"1,164.36",27.21,"389,577"
UX/UI Design,"19,815",1.10%,218,"1,173",221,"259,300",1.01,"149,523",685.887385,7.55,"1,189.45",13.09,"109,777"
Web Developer,"19,815",0.68%,135,"1,047",137,"143,450",1.01,"149,523",1107.581111,7.55,"1,062.59",7.24,"-6,073"


In [114]:
# Анализ чувствительности ПО ПРОДУКТАМ (±10% для каждого)
# Изменяем рычаги: UA, C1, APC, AOV, LTC (CPA)

delta = 0.10
sens_results = []

# Берем топ-5 продуктов по выручке для анализа
top_products = product_ue.head(5).index.tolist()

for prod in top_products:
    row = product_ue.loc[prod]
    # Базовые значения
    ua0   = row['UA']
    c10   = row['c1']
    apc0  = row['apc']
    aov0  = row['aov']
    ltc0  = row['cpa']
    
    # Сценарии (изменения только ОДНОЙ метрики за раз)
    scenarios = [
        ('1. Факт (база)',  ua0, c10, apc0, aov0, ltc0),
        ('2. +10% UA',      ua0 * (1 + delta), c10, apc0, aov0, ltc0),
        ('3. +10% C1',      ua0, c10 * (1 + delta), apc0, aov0, ltc0),
        ('4. +10% APC',     ua0, c10, apc0 * (1 + delta), aov0, ltc0),
        ('5. +10% AOV',     ua0, c10, apc0, aov0 * (1 + delta), ltc0),
        ('6. -10% LTC (CPA)', ua0, c10, apc0, aov0, ltc0 * (1 - delta)),
    ]
    
    for label, ua, c1, apc, aov, ltc in scenarios:
        # Расчет в глубину
        b_new    = ua * c1
        t_new    = b_new * apc
        rev_new  = t_new * aov
        ac_new   = ua * ltc
        cltv_new = apc * aov
        ltv_new  = c1 * cltv_new
        cm_new   = rev_new - ac_new
        
        sens_results.append({
            'Продукт':   prod,
            'Сценарий':  label,
            'UA':        ua,
            'C1':        c1,
            'APC':       apc,
            'AOV':       aov,
            'LTC':       ltc,
            'CLTV, €':   cltv_new,
            'LTV, €':    ltv_new,
            'Rev, €':    rev_new,
            'AC, €':     ac_new,
            'CM, €':     cm_new
        })

sens_prod_df = pd.DataFrame(sens_results)

# Считаем ΔCM относительно базы
final_list = []
for prod, group in sens_prod_df.groupby('Продукт', observed=True):
    base_cm = group.iloc[0]['CM, €']
    group['ΔCM, €'] = group['CM, €'] - base_cm
    group.iloc[0, group.columns.get_loc('ΔCM, €')] = np.nan
    final_list.append(group)

sens_prod_df = pd.concat(final_list)

display(
    sens_prod_df.set_index(['Продукт', 'Сценарий']).style
    .format({
        'UA':        '{:,.0f}',
        'C1':        '{:.2%}',
        'APC':       '{:.2f}',
        'AOV':       '{:,.0f}',
        'LTC':       '{:.2f}',
        'CLTV, €':   '{:,.2f}',
        'LTV, €':    '{:,.2f}',
        'Rev, €':    '{:,.0f}',
        'AC, €':     '{:,.0f}',
        'CM, €':     '{:,.0f}',
        'ΔCM, €':    '{:+,.0f}',
    }, na_rep='—')
    .map(
        lambda v: 'color: red' if isinstance(v, (int, float)) and v < 0 else '',
        subset=['CM, €']
    )
    .set_caption(f'Анализ чувствительности: Рычаги роста (±{delta:.0%})')
)


#### Рассматривать продукты отдельно не имеет смысла, потому что в большинстве случаев отделить лидов по продукту не получится. При этом влияние метрик будет аналогичным.

## Гипотезы и A/B тесты

────────────────────────────────────────────────────────────────────────

[H1] А/В Тест (Сокращение времени первого ответа [SLA] до уровня
     Q1 увеличит конверсию лида в покупателя более чем на 10%):

  Для проверки утверждения мы разделим входящие лиды на две группы А и В 
  с максимально похожим составом (источник, качество лида).
  Группа А — текущий процесс обработки лида (стандартное SLA).
  Группа В — приоритет ответа ≤ Q1 (50мин) SLA (назначение быстрейшим менеджерам).
  Гипотезу считаем подтверждённой, если в группе В C1 будет
  в пределах 4.52% - 5%.
  Риск: необходима рандомизация назначения лида менеджеру.

────────────────────────────────────────────────────────────────────────

[H2] А/В Тест (Акцент на высокомаржинальных продуктах в оффере
     увеличит средний чек покупателя не менее чем на 10%):
  Трафик делится на группы А и В — одинаковый объём, разный оффер.
  Группа А — текущий продуктовый микс (офферы как есть).
  Группа В — топ продукт по марже выделен первым в оффере.
  Гипотезу считаем подтверждённой, если средний чек группы В превысит
  1,273 € (до 1400 € текущий 1,157 € + 10%).
  Риск: необходимо разделение трафика без пересечения групп.

────────────────────────────────────────────────────────────────────────

[H3] А/В Тест (Приоритет обработки лидов quality=high лучшими
     менеджерами (C1 > медианы) увеличит конверсию на 15%):
  Для проверки утверждения мы разделим лиды на две группы А и В
  с максимально похожим составом.
  Группа А — равномерное распределение лидов на всех менеджеров.
  Группа В — quality=high/excellent → менеджеры с C1 > медианы по команде.
  Гипотезу считаем подтверждённой, если в группе В C1 превысит
  4.72%.
  Риск: потеря выручки от middle-лидов; контроль общего объёма;
    конфликт со стороны персонала.
────────────────────────────────────────────────────────────────────────

In [123]:
# План A/B-тестирования: По одной уникальной гипотезе для каждого ТОП-продукта

alpha = 0.05
power = 0.8
delta = 0.15 # Ожидаемый относительный прирост (MDE)

# Определяем специфичные гипотезы для выбранных продуктов:
# 1. Digital Marketing: Акцент на трудоустройстве и зарплате (ROI обучения)
# 2. UX/UI Design: Визуальный лид-магнит (бесплатный мини-воркшоп по Figma)
# 3. Web Developer: Технический скрининг (бесплатная проверка кода/консультация)

product_hypotheses = [
    {
        "Продукт": "Digital Marketing",
        "Гипотеза": "[H_DM] Добавление в оффер калькулятора ROI обучения (зарплата через 6 мес) увеличит C1 на 15%",
        "Baseline (C1)": product_ue.loc["Digital Marketing", "c1"] if "Digital Marketing" in product_ue.index else crv_rate,
        "MDE": delta,
        "Тип": "ROI Calc"
    },
    {
        "Продукт": "UX/UI Design",
        "Гипотеза": "[H_UX] Замена вводного звонка на мини-воркшоп 'Первый проект в Figma' увеличит C1 на 15%",
        "Baseline (C1)": product_ue.loc["UX/UI Design", "c1"] if "UX/UI Design" in product_ue.index else crv_rate,
        "MDE": delta,
        "Тип": "Workshop"
    },
    {
        "Продукт": "Web Developer",
        "Гипотеза": "[H_Web] Предложение бесплатного технического аудита (code review) на первом этапе увеличит C1 на 15%",
        "Baseline (C1)": product_ue.loc["Web Developer", "c1"] if "Web Developer" in product_ue.index else crv_rate,
        "MDE": delta,
        "Тип": "Code Review"
    }
]

# Расчет объема выборки для каждой гипотезы
ab_product_results = []
# Среднее кол-во лидов в день на ВЕСЬ проект (т.к. лиды общие)
daily_leads_global = n_leads / ((deals['created_time'].max() - deals['created_time'].min()).days + 1)

for h_data in product_hypotheses:
    p1 = h_data["Baseline (C1)"]
    p2 = p1 * (1 + h_data["MDE"])
    
    n_per_group = h.ab_sample_size(p1, h_data["MDE"], alpha=alpha, power=power)
    n_total = n_per_group * 2
    days_needed = n_total / daily_leads_global
    
    ab_product_results.append({
        "Продукт": h_data["Продукт"],
        "Гипотеза": h_data["Гипотеза"],
        "Базовый C1": f"{p1:.2%}",
        "Целевой C1": f"{p2:.2%}",
        "Выборка (всего)": int(n_total),
        "Дней теста": f"{round(days_needed, 0):.0f}",
        "Реализуемо за 21д?": "Да" if days_needed <= 21 else "Нет"
    })

ab_comparison_df = pd.DataFrame(ab_product_results)

print(f"План тестирования продуктовых гипотез (Бизнес-модель одной продажи, MDE={delta*100:.0f}%)")
display(ab_comparison_df.style.set_caption("Сравнение длительности тестов по продуктовым гипотезам"))


План тестирования продуктовых гипотез (Бизнес-модель одной продажи, MDE=15%)


,Продукт,Гипотеза,Базовый C1,Целевой C1,Выборка (всего),Дней теста,Реализуемо за 21д?
0,Digital Marketing,[H_DM] Добавление в оффер калькулятора ROI обучения (зарплата через 6 мес) увеличит C1 на 15%,2.34%,2.69%,62510,1117,Нет
1,UX/UI Design,[H_UX] Замена вводного звонка на мини-воркшоп 'Первый проект в Figma' увеличит C1 на 15%,1.10%,1.27%,134568,2404,Нет
2,Web Developer,[H_Web] Предложение бесплатного технического аудита (code review) на первом этапе увеличит C1 на 15%,0.68%,0.78%,218290,3900,Нет


### Проблема скорости тестов (Низкий трафик)

Если мы продолжим измерять «Деньги» (финальную оплату с C1 ≈ 4%), то при текущем трафике (~56 лидов/день) тест с MDE 10% будет идти 1371 день (3.5 года). Это слишком долго.

> **Решение:** Переход от финальных метрик (Деньги) к прокси-метрикам (Действия) с более высокой базовой конверсией.

### Новые гипотезы 

#### [H4] Автоматизация первого касания
- **Суть:** Внедрение WhatsApp-бота, который пишет лиду в течение 2 минут после регистрации.
- **Прокси-метрика:** Конверсия в «Успешный контакт / Дозвон» (ожидаем рост с 50% до 65%).
- **Зачем:** Нулевой SLA гарантированно повышает лояльность и вероятность продажи.

#### [H5] Лид-магнит и фокус на Интенсив 
- **Суть:** Вместо продажи курса «в лоб», предлагать бесплатный интенсив/пробные уроки.
- **Прокси-метрика:** Конверсия в «Запись на интенсив» (ожидаем базовую конверсию ~30%).
- **Зачем:** Снимаем барьер первого платежа. 

---
#### Расчет времени проверки гипотез:


In [124]:
# Сравнение времени теста для разных гипотез
p_h1 = 0.041
p_h4 = 0.50          # Прокси: успешный контакт
p_h5 = 0.30          # Прокси: запись на бесплатный продукт

mde_h1 = 0.10        # Ожидаем +10% роста оплат
mde_h4 = 0.40        # Ожидаем +40% роста дозвона (автоматизация) 
mde_h5 = 0.50        # Ожидаем +50% записи (бесплатный продукт вместо платного)

# Используем ранее рассчитанный daily_leads_global
plans = []
for label, p, mde in [
    ("H1: SLA (Оплата)", p_h1, mde_h1),
    ("H4: Auto-touch (Контакт)", p_h4, mde_h4), 
    ("H5: Lead-Magnet (Запись)", p_h5, mde_h5)
]:
    n_group = h.ab_sample_size(p, mde)
    n_total = n_group * 2
    # Используем GLOBAL traffic (все 56 лидов в день на тест)
    days = n_total / daily_leads_global
    
    plans.append({
        "Гипотеза": label,
        "Базовая конф.": f"{p:.1%}",
        "MDE (отн.)": f"+{mde:.0%}",
        "Цель": f"{p*(1+mde):.1%}",
        "Выборка (всего)": int(n_total),
        "Дней теста": round(days, 1),
        "Реализуемо за 14д?": " ДА" if days <= 14 else " НЕТ"
    })

comparison_df = pd.DataFrame(plans)
print(comparison_df.to_string(index=False))
display(comparison_df.style.set_caption("Сравнение длительности тестов: Деньги против действий"))


                Гипотеза Базовая конф. MDE (отн.)  Цель  Выборка (всего)  Дней теста Реализуемо за 14д?
        H1: SLA (Оплата)          4.1%       +10%  4.5%            76900     1373.80                НЕТ
H4: Auto-touch (Контакт)         50.0%       +40% 70.0%              186        3.30                 ДА
H5: Lead-Magnet (Запись)         30.0%       +50% 45.0%              324        5.80                 ДА


,Гипотеза,Базовая конф.,MDE (отн.),Цель,Выборка (всего),Дней теста,Реализуемо за 14д?
0,H1: SLA (Оплата),4.1%,+10%,4.5%,76900,1373.800000,НЕТ
1,H4: Auto-touch (Контакт),50.0%,+40%,70.0%,186,3.300000,ДА
2,H5: Lead-Magnet (Запись),30.0%,+50%,45.0%,324,5.800000,ДА


In [117]:
# Расчет воронки и конверсии по SLA для отчета
funnel = deals['stage'].value_counts().to_frame().rename(columns={'count': 'leads'})

# Конверсия в зависимости от SLA (генерация данных, если они не были созданы ранее)
if 'sla_group' in deals.columns:
    sla_cr = deals.groupby('sla_group', observed=True)['is_buyer'].mean().to_frame().rename(columns={'is_buyer': 'cr'})
else:
    # Заглушка, если колонка не найдена (предотвращает ошибку в финальной ячейке)
    sla_cr = pd.DataFrame({'cr': [crv_rate]}, index=['Global'])

if 'quality' in deals.columns:
    q_cr = deals.groupby('quality', observed=True)['is_buyer'].mean().to_frame().rename(columns={'is_buyer': 'cr'})
else:
    q_cr = pd.DataFrame()


In [125]:
#  Сохранение данных для презентации (08_presentation.ipynb) 
# Все ключевые агрегаты пакуем в один словарь → report_data.pkl

REPORT_DATA_PATH = os.path.join(CLEANED_DIR, 'report_data.pkl')

# Гипотезы по продуктам для отчета
hypotheses_v3 = ab_comparison_df.to_dict('records')

report_data = {
    # Глобальные KPI
    'global_kpi': {
        'n_leads':       n_leads,
        'n_buyers':      n_buyers,
        'crv_rate':      crv_rate,
        'avg_ltv':       ltv,    
        'cac':           cac,
        'total_revenue': total_revenue,
        'total_spend':   total_spend,
        'profit':        profit,
        'roas':          total_revenue / total_spend,
        'apc':           apc,
        'cltv':          cltv,
        'ltv':           ltv,
        'cpa':           ltc
    },

    # Таблицы юнит-экономики
    'source_ue':  source_ue.reset_index(drop=True), # source_ue должен быть рассчитан выше
    'product_ue': product_ue.head(15).reset_index(),

    # Анализ чувствительности (Рычаги роста)
    'sens_df': sens_prod_df.reset_index(),

    # Воронка и гипотезы
    'funnel': funnel.reset_index(),
    'sla_cr': sla_cr.reset_index(),
    'q_cr':   q_cr.reset_index() if 'q_cr' in locals() else pd.DataFrame(),

    # Продуктовые гипотезы
    'hypotheses': hypotheses_v3,
}

os.makedirs(CLEANED_DIR, exist_ok=True)
pd.to_pickle(report_data, REPORT_DATA_PATH)

print(f'✓ Данные успешно сохранены в: {REPORT_DATA_PATH}')
print(f'Передано {len(report_data["hypotheses"])} продуктовых гипотез для финального отчета.')


✓ Данные успешно сохранены в: ../data/cleaned/report_data.pkl
Передано 3 продуктовых гипотез для финального отчета.


## Итоги продуктовой аналитики

1.  **Бизнес-модель**: Подтверждено, что это модель «одной продажи» ($T/B \approx 1.01$). Основной фокус должен быть на **первичной конверсии (C1)**.
2.  **Точки роста**: 
    - Максимальный рычаг дает увеличение $C1$. Прирост конверсии на 10% увеличивает маржинальную прибыль значительнее, чем аналогичное масштабирование трафика (из-за фиксированных затрат на лид).
3.  **A/B тестирование**: Тестирование «денежных» метрик (C1 в оплату) при текущем трафике нецелесообразно (тест займет годы). Необходимо переходить к **прокси-метрикам** (дозвон, запись на пробный урок), где время теста составит **7–14 дней**.
4.  **Рекомендация**: Внедрение Лид-магнит (бесплатного интенсива) — самая перспективная гипотеза для быстрого масштабирования базы клиентов без кратного увеличения рекламного бюджета. Можно параллельно с "Автокасанием" (Внедрение WhatsApp/Telegram-бота, который пишет лиду в течение 2 минут после регистрации.)